# Guia - SQL

Esta está diseñado para que practiques consultas SQL usando `ipython-sql` y una base de datos sobre entidades en una **universidad**: estudiantes, carreras, cursos, profesores, semestres, secciones e inscripciones.

## Objetivos del taller

- Crear tablas con claves primarias y foráneas.
- Insertar datos en una base SQLite.
- Realizar consultas básicas y avanzadas con SQL.
- Aplicar funciones de agregación y subconsultas.
- Practicar `JOIN` entre varias tablas.

## Esquema de la base de datos

Trabajaremos con el siguiente esquema:

- `Carreras(id_carrera, nombre_carrera)`
- `Estudiantes(id_est, nombre, fecha_nac, id_carrera)`
- `Profesores(id_prof, nombre_prof, categoria)`
- `Cursos(id_curso, nombre_curso, creditos)`
- `Semestres(id_sem, anio, periodo)`
- `Secciones(id_sec, id_curso, id_prof, id_sem, tipo)`
- `Inscripciones(id_insc, id_est, id_sec, nota_final, asistencia)`


## Conexión a SQLite

**Ojo**: cada vez que anteponemos `%sql` es porque esa línea corresponde a un comando a SQL que va a la base de datos en la que estamos trabajando (en este caso `universidad.db`). Si queremos tener todo un bloque con instrucciones SQL tenemos que usar `%%sql` (habrán ejemplos de esto a lo largo del _notebook_).

In [1]:
%load_ext sql
%sql sqlite:///universidad.db

## Crear tablas

### Sintaxis general

Para crear una tabla con clave primaria:

```sql
CREATE TABLE NombreTabla (
    atributo_1 TIPO,
    atributo_2 TIPO,
    ...,
    PRIMARY KEY(atributo_1)
);
```

Para incluir claves foráneas:

```sql
CREATE TABLE NombreTabla (
    atributo_1 TIPO,
    atributo_2 TIPO,
    atributo_fk TIPO,
    ...,
    PRIMARY KEY(atributo_1),
    FOREIGN KEY (atributo_fk) REFERENCES OtraTabla(atributo_referido)
);
```

### Creación del esquema

A continuación se muestran las sentencias para crear todas las tablas de nuestro esquema académico.


In [2]:
%%sql

-- Borramos las tablas si existen, en orden inverso a las dependencias
DROP TABLE IF EXISTS Inscripciones;
DROP TABLE IF EXISTS Secciones;
DROP TABLE IF EXISTS Semestres;
DROP TABLE IF EXISTS Cursos;
DROP TABLE IF EXISTS Profesores;
DROP TABLE IF EXISTS Estudiantes;
DROP TABLE IF EXISTS Carreras;

-- Tabla de carreras
CREATE TABLE Carreras (
    id_carrera     INTEGER PRIMARY KEY,
    nombre_carrera TEXT
);

-- Estudiantes pertenecen a una carrera
CREATE TABLE Estudiantes (
    id_est     INTEGER PRIMARY KEY,
    nombre     TEXT,
    fecha_nac  TEXT,         -- formato 'YYYY-MM-DD'
    id_carrera INTEGER,
    FOREIGN KEY (id_carrera) REFERENCES Carreras(id_carrera)
);

-- Profesores
CREATE TABLE Profesores (
    id_prof     INTEGER PRIMARY KEY,
    nombre_prof TEXT,
    categoria   TEXT
);

-- Cursos
CREATE TABLE Cursos (
    id_curso     INTEGER PRIMARY KEY,
    nombre_curso TEXT,
    creditos     INTEGER
);

-- Semestres (por ejemplo anio=2024, periodo='1')
CREATE TABLE Semestres (
    id_sem  INTEGER PRIMARY KEY,
    anio    INTEGER,
    periodo TEXT
);

-- Secciones de cursos en un semestre, dictadas por un profesor
CREATE TABLE Secciones (
    id_sec   INTEGER PRIMARY KEY,
    id_curso INTEGER,
    id_prof  INTEGER,
    id_sem   INTEGER,
    tipo     TEXT, -- 'Teórica', 'Laboratorio', etc.
    FOREIGN KEY (id_curso) REFERENCES Cursos(id_curso),
    FOREIGN KEY (id_prof)  REFERENCES Profesores(id_prof),
    FOREIGN KEY (id_sem)   REFERENCES Semestres(id_sem)
);

-- Inscripciones de estudiantes a secciones, con nota y asistencia
CREATE TABLE Inscripciones (
    id_insc    INTEGER PRIMARY KEY,
    id_est     INTEGER,
    id_sec     INTEGER,
    nota_final REAL,
    asistencia REAL, -- proporción entre 0 y 1
    FOREIGN KEY (id_est) REFERENCES Estudiantes(id_est),
    FOREIGN KEY (id_sec) REFERENCES Secciones(id_sec)
);

 * sqlite:///universidad.db
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.


[]

Si ejecutamos la consulta `SELECT * FROM Carreras` notaremos que el resultado es vacío. Esto es evidente, ya que esta consulta me retorna todo lo que tengo en la tabla `Carreras` y actualmente no hemos insertado nada.

In [3]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'
%sql Select * from Carreras;

 * sqlite:///universidad.db
Done.


id_carrera,nombre_carrera


## Insertar valores

### Forma básica de `INSERT`

La sintaxis básica es:

```sql
INSERT INTO Tabla(atributo_1, atributo_2, ...)
VALUES (valor_1, valor_2, ...);
```

Si se listan **todos** los atributos en el mismo orden definido en `CREATE TABLE`, se puede omitir la lista de columnas:

```sql
INSERT INTO Tabla
VALUES (valor_1, valor_2, ...);
```

### Ejemplo sencillo


In [4]:
%%sql
INSERT INTO Carreras VALUES (1, 'Ingeniería Informática');

SELECT * FROM Carreras;

 * sqlite:///universidad.db
1 rows affected.
Done.


id_carrera,nombre_carrera
1,Ingeniería Informática


## Poblar las tablas con datos

Ahora vamos a poblar todas las tablas con un conjunto de datos de ejemplo.


In [5]:
%%sql

-- Carreras
INSERT INTO Carreras VALUES
(2,'Ingeniería Civil'),
(3,'Psicología'),
(4,'Diseño');

-- Estudiantes
INSERT INTO Estudiantes VALUES
(1,'María López','2002-03-15',1),
(2,'Juan Pérez','1999-11-02',2),
(3,'Ana Torres','2004-07-22',3),
(4,'Luis González','1998-01-30',1),
(5,'Sofía Rivas','2005-09-10',4),
(6,'Pedro Sánchez','2001-05-05',2);

-- Profesores
INSERT INTO Profesores VALUES
(1,'Carlos Núñez','Titular'),
(2,'Elena Ruiz','Asociado'),
(3,'Miguel Herrera','Adjunto');

-- Cursos
INSERT INTO Cursos VALUES
(1,'Bases de Datos',10),
(2,'Introducción a la programación',8),
(3,'Cálculo I',9),
(4,'Psicología General',7);

-- Semestres
INSERT INTO Semestres VALUES
(1,2023,'1'),
(2,2023,'2'),
(3,2024,'1'),
(4,2024,'2');

-- Secciones
INSERT INTO Secciones VALUES
(1,1,1,3,'Teórica'),      -- Bases de Datos, 2024-1, Carlos
(2,2,2,3,'Teórica'),      -- Introducción a la programación, 2024-1, Elena
(3,1,1,4,'Teórica'),      -- Bases de Datos, 2024-2, Carlos
(4,3,2,1,'Teórica'),      -- Cálculo I, 2023-1, Elena
(5,4,3,2,'Teórica');      -- Psicología General, 2023-2, Miguel

-- Inscripciones
INSERT INTO Inscripciones VALUES
(1,1,1,6.5,0.95),  -- María en BD 2024-1
(2,2,1,5.0,0.80),  -- Juan en BD 2024-1
(3,3,2,5.8,0.90),  -- Ana en Prog I 2024-1
(4,4,3,6.8,0.92),  -- Luis en BD 2024-2
(5,1,3,6.2,0.88),  -- María en BD 2024-2
(6,5,5,5.5,0.85),  -- Sofía en Psico 2023-2
(7,6,4,4.9,0.75),  -- Pedro en Cálculo 2023-1
(8,3,5,6.0,0.90);  -- Ana en Psico 2023-2


 * sqlite:///universidad.db
3 rows affected.
6 rows affected.
3 rows affected.
4 rows affected.
4 rows affected.
5 rows affected.
8 rows affected.


[]

### Para verificar que los datos se insertaron correctamente

In [6]:
%%sql
SELECT * FROM Estudiantes;

 * sqlite:///universidad.db
Done.


id_est,nombre,fecha_nac,id_carrera
1,María López,2002-03-15,1
2,Juan Pérez,1999-11-02,2
3,Ana Torres,2004-07-22,3
4,Luis González,1998-01-30,1
5,Sofía Rivas,2005-09-10,4
6,Pedro Sánchez,2001-05-05,2


In [7]:
%%sql
SELECT * FROM Secciones;

 * sqlite:///universidad.db
Done.


id_sec,id_curso,id_prof,id_sem,tipo
1,1,1,3,Teórica
2,2,2,3,Teórica
3,1,1,4,Teórica
4,3,2,1,Teórica
5,4,3,2,Teórica


In [8]:
%%sql
SELECT * FROM Inscripciones;

 * sqlite:///universidad.db
Done.


id_insc,id_est,id_sec,nota_final,asistencia
1,1,1,6.5,0.95
2,2,1,5.0,0.8
3,3,2,5.8,0.9
4,4,3,6.8,0.92
5,1,3,6.2,0.88
6,5,5,5.5,0.85
7,6,4,4.9,0.75
8,3,5,6.0,0.9


## Ejecución de consultas

Las consultas más básicas son de la forma `SELECT - FROM - WHERE`. En general, la consulta de álgebra relacional:

$$
\pi_{a_1, \dots, a_n}(\sigma_{\text{condiciones}}(R_1 \times R_m))
$$

se traduce en SQL como:

```SQL
SELECT a_1, ..., a_n
FROM R_1, ..., R_m
WHERE <condiciones>
```

Por ejemplo:

```sql
SELECT columnas
FROM tablas
WHERE condiciones;
```

Y para ver un estudiante específico:


In [9]:
%%sql
SELECT *
FROM Estudiantes
WHERE id_est = 1;

 * sqlite:///universidad.db
Done.


id_est,nombre,fecha_nac,id_carrera
1,María López,2002-03-15,1


### JOIN entre tablas

Para unir información de varias tablas, usamos `JOIN`. Por ejemplo, estudiantes con el nombre de su carrera:


In [10]:
%%sql
SELECT e.nombre AS estudiante,
       c.nombre_carrera
FROM Estudiantes AS e
JOIN Carreras    AS c ON e.id_carrera = c.id_carrera;

 * sqlite:///universidad.db
Done.


estudiante,nombre_carrera
María López,Ingeniería Informática
Juan Pérez,Ingeniería Civil
Ana Torres,Psicología
Luis González,Ingeniería Informática
Sofía Rivas,Diseño
Pedro Sánchez,Ingeniería Civil


Otro ejemplo, secciones con su curso y semestre:

In [11]:
%%sql
SELECT s.id_sec,
       c.nombre_curso,
       se.anio,
       se.periodo
FROM Secciones AS s
JOIN Cursos    AS c  ON s.id_curso = c.id_curso
JOIN Semestres AS se ON s.id_sem   = se.id_sem;

 * sqlite:///universidad.db
Done.


id_sec,nombre_curso,anio,periodo
1,Bases de Datos,2024,1
2,Introducción a la programación,2024,1
3,Bases de Datos,2024,2
4,Cálculo I,2023,1
5,Psicología General,2023,2


## Ejercicios

Responde los siguientes ejercicios usando consultas SQL.


### Ejercicio 1

Listar todos los cursos que se dictan en el semestre **2024-1**.


In [12]:
%%sql



 * sqlite:///universidad.db


### Ejercicio 2

Listar todas las y los estudiantes **nacidos después del año 2003**. Puede resultar util la funcion [SUBSTR](ttps://www.w3schools.com/sql/func_mysql_substr.asp).


In [13]:
%%sql



 * sqlite:///universidad.db


### Ejercicio 3

Insertar la carrera `Ingeniería de Datos` en la tabla `Carreras`.


In [14]:
%%sql



 * sqlite:///universidad.db


### Ejercicio 4

Crear una sección `Laboratorio` del curso `Bases de Datos` en el semestre **2024-2** a cargo de la profesora `Elena Ruiz`.

Supón que:
- El curso `Bases de Datos` tiene `id_curso = 1`.
- La profesora `Elena Ruiz` tiene `id_prof = 2`.
- El semestre 2024-2 tiene `id_sem = 4`.
- Puedes usar `id_sec = 6` para la nueva sección.


In [15]:
%%sql



 * sqlite:///universidad.db


### Ejercicio 5

Inscribir a la estudiante `Sofía Rivas` en la nueva sección de laboratorio de `Bases de Datos` con nota final **6.0** y asistencia **100%**.

Supón que:
- `Sofía Rivas` tiene `id_est = 5`.
- La sección de laboratorio creada en el ejercicio 4 tiene `id_sec = 6`.
- Usa `id_insc = 9` para la nueva inscripción.


In [16]:
%%sql



 * sqlite:///universidad.db


### Ejercicio 6

Listar todas las y los estudiantes con el nombre de su carrera.


In [17]:
%%sql



 * sqlite:///universidad.db


### Ejercicio 7

Mostrar todas las secciones del curso `Bases de Datos` con el nombre del profesor y el semestre (año y período).


In [18]:
%%sql



 * sqlite:///universidad.db


### Ejercicio 8

Listar los nombres de las carreras que tienen estudiantes inscritos en el semestre **2023-2**.


In [19]:
%%sql



 * sqlite:///universidad.db


### Ejercicio 9

Mostrar el nombre de la o el estudiante con la **mayor nota final**. Puede resultar util la funcion [MAX](https://www.w3schools.com/sql/sql_min_max.asp).


In [20]:
%%sql



 * sqlite:///universidad.db


### Ejercicio 10

Contar cuántos estudiantes están inscritos en cada curso. Puede resultar util la funcion [COUNT](https://www.w3schools.com/sql/sql_min_max.asp).


In [21]:
%%sql



 * sqlite:///universidad.db


### Ejercicio 11

Listar los semestres en los que ha estado inscrita la estudiante `María López`.


In [22]:
%%sql



 * sqlite:///universidad.db


### Ejercicio 12

Listar todas las inscripciones junto con el nombre del curso, nombre de la o el estudiante, nombre del profesor, año, período y nota final.


In [23]:
%%sql



 * sqlite:///universidad.db
